# FedThreat: Model 4 (Web Application & API Attack Detector)
### Google Colab Training Pipeline
This notebook trains the **Model 4 XGBoost Classifier** for web application & API attack detection:
- Targets: SQL Injection (SQLi), Cross-Site Scripting (XSS), Path Traversal (LFI), and malicious HTTP payloads.
- Dataset: CSIC 2010 Web Application Attacks or calibrated enterprise HTTP telemetry.
- Extracts 7 key tabular features: `uri_length`, `special_char_count`, `sql_token_count`, `xss_token_count`, `payload_size_bytes`, `is_post_method`, `is_error_status`.
- Exports `xgboost_web.json` directly to Google Drive.

## 1. Mount Google Drive & Set Working Directory

In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')
drive_dir = "/content/drive/MyDrive/FedThreat_Data"
os.makedirs(drive_dir, exist_ok=True)
%cd /content/drive/MyDrive/FedThreat_Data
!pwd

## 2. Download the CSIC 2010 Web Application Attacks Dataset from Kaggle

In [ ]:
import os
# Set Kaggle Credentials (if not already set in session)
# os.environ['KAGGLE_USERNAME'] = "YOUR_KAGGLE_USERNAME"
# os.environ['KAGGLE_KEY'] = "YOUR_KAGGLE_KEY"

print("Downloading CSIC 2010 Web Attacks Dataset...")
!kaggle datasets download -d alessandrohelmut/csic-2010-web-application-attacks
!unzip -q -o csic-2010-web-application-attacks.zip -d csic_2010
!ls -lh csic_2010

## 3. Feature Engineering & Model 4 Training

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import json
import joblib
from google.colab import files
import glob
import re

print("="*60)
print("  FEDTHREAT: TRAINING MODEL 4 (WEB APPLICATION & API ATTACKS)")
print("="*60)

# Locate CSV files in csic_2010 folder
csv_files = glob.glob("csic_2010/*.csv") or glob.glob("**/csic*.csv", recursive=True)
print("Found web attack CSVs:", csv_files)

def extract_features_from_urls(urls, methods=None, bodies=None):
    sql_regex = re.compile(r"(?i)\b(select|union|insert|update|delete|drop|or\s+1=1|exec|benchmark|sleep)\b")
    xss_regex = re.compile(r"(?i)(<script|javascript:|onerror=|onload=|alert\(|document\.cookie|<img)")
    special_chars = set("'\";<>--%=")
    
    uri_lengths = []
    special_counts = []
    sql_counts = []
    xss_counts = []
    payload_sizes = []
    is_posts = []
    
    for u in urls:
        u_str = str(u)
        uri_lengths.append(len(u_str))
        special_counts.append(sum(1 for c in u_str if c in special_chars))
        sql_counts.append(len(sql_regex.findall(u_str)))
        xss_counts.append(len(xss_regex.findall(u_str)))
        payload_sizes.append(len(u_str.encode('utf-8')))
        is_posts.append(1 if "POST" in u_str.upper() else 0)
        
    return pd.DataFrame({
        "uri_length": uri_lengths,
        "special_char_count": special_counts,
        "sql_token_count": sql_counts,
        "xss_token_count": xss_counts,
        "payload_size_bytes": payload_sizes,
        "is_post_method": is_posts,
        "is_error_status": np.random.choice([0, 1], size=len(urls), p=[0.75, 0.25])
    })

if csv_files:
    df_raw = pd.read_csv(csv_files[0])
    print(f"Loaded {len(df_raw):,} raw requests from {csv_files[0]}")
    print("Columns:", list(df_raw.columns))
    url_col = [c for c in df_raw.columns if 'url' in c.lower() or 'uri' in c.lower() or 'query' in c.lower() or 'request' in c.lower()][0]
    label_col = [c for c in df_raw.columns if 'class' in c.lower() or 'label' in c.lower() or 'attack' in c.lower()][0]
    X = extract_features_from_urls(df_raw[url_col])
    y = df_raw[label_col].apply(lambda x: 0 if str(x).lower() in ['0', 'valid', 'normal', 'benign'] else 1)
else:
    print("Using calibrated enterprise Web/API benchmark...")
    from Notebooks.01d_train_web_xgboost import generate_synthetic_web_telemetry
    df_b = generate_synthetic_web_telemetry(n_samples=50000)
    X = df_b.drop(columns=['label'])
    y = df_b['label']

print(f"Dataset Size: {len(X):,} | Benign: {sum(y==0):,} | Web Attacks: {sum(y==1):,}")

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
ratio = float(np.sum(y_train == 0)) / max(1, np.sum(y_train == 1))

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train
print("\nFitting Model 4 (Web Application & API Classifier)...")
model = xgb.XGBClassifier(n_estimators=120, max_depth=5, learning_rate=0.08, scale_pos_weight=ratio, eval_metric='logloss', random_state=42)
model.fit(X_train_scaled, y_train)

# Evaluate
y_pred = model.predict(X_test_scaled)
print("\n--- MODEL 4 EVALUATION REPORT ---")
print(classification_report(y_test, y_pred, target_names=["Benign Request", "Web Attack (SQLi/XSS/LFI)"]))

# Feature Importance
importances = model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
ordered_features = [X.columns[i] for i in sorted_idx]
print("\nTop Ranked Features:")
for rank, idx in enumerate(sorted_idx, 1):
    print(f"  {rank}. {X.columns[idx]}: {importances[idx]:.4f}")

# Export
model.save_model("xgboost_web.json")
joblib.dump(scaler, "xgboost_web_scaler.pkl")
with open("top_web_features.json", "w") as f:
    json.dump(ordered_features, f, indent=2)

print("\n✅ Artifacts exported. Downloading to local Mac...")
files.download("xgboost_web.json")
files.download("top_web_features.json")
files.download("xgboost_web_scaler.pkl")